# Note
Change `file_path` to your file location where `articles_metadata.csv` is at.

In [1]:
import os
import csv
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
def get_article_text(link, file_path):
    """Fetches article text, handles variations, and saves to file."""
    retries = 3
    for attempt in range(retries):
        driver = None
        try:
            driver = webdriver.Chrome()
            driver.get(link)

            try:
                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, '._article_content')))
            except:
                print(f"Timed out waiting for content on {link} (attempt {attempt+1})")
                continue

            soup = BeautifulSoup(driver.page_source, "html.parser")
            content_div = soup.select_one("._article_content")

            if not content_div:
                print(f"Could not find _article_content div on {link} (attempt: {attempt+1})")
                continue

            article_text = content_div.get_text(strip=True, separator='\n')

            if article_text:
                directory = os.path.dirname(file_path)
                
                # If a directory path exists and the folder is missing, create it
                if directory and not os.path.exists(directory):
                    os.makedirs(directory, exist_ok=True)

                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(article_text)
                return
            else:
                print(f"No article text found for {link} (attempt: {attempt+1})")

        except Exception as e:
            print(f"Error processing {link} (attempt: {attempt+1}) {e}")
            time.sleep(5)

        finally:
            if driver:
                driver.quit()

    print(f"Failed to process {link} after {retries} retries.")

In [3]:
def main():
    file_path = "../data/articles_metadata.csv"
    article_data = []
    try:  
        with open(file_path, "r", encoding="utf-8") as csvfile:
            reader = csv.DictReader(csvfile) 
            for row in reader:
                article_data.append((row["link"], row["file_path"]))
    except FileNotFoundError:
        print(f"Error: CSV file not found at {file_path}")
        return  
    except Exception as e: 
        print(f"Error reading CSV file: {e}")
        return

    with ThreadPoolExecutor(max_workers=8) as executor:  
        futures = {executor.submit(get_article_text, link, file_path): link for link, file_path in article_data}

        for future in as_completed(futures):
            link = futures[future]
            if future.exception() is not None:
                print(f"An error occurred while processing {link}: {future.exception()}")
            else:
                print(f"Finished processing {link}")

if __name__ == "__main__":
    main()

Finished processing https://m.entertain.naver.com/ranking/article/311/0001911507
Finished processing https://m.entertain.naver.com/ranking/article/117/0003982512
Finished processing https://m.entertain.naver.com/ranking/article/241/0003463184
Finished processing https://m.entertain.naver.com/ranking/article/076/0004318319
Finished processing https://m.entertain.naver.com/ranking/article/312/0000725821
Finished processing https://m.entertain.naver.com/ranking/article/468/0001173807
Finished processing https://m.entertain.naver.com/ranking/article/108/0003361494
Finished processing https://m.entertain.naver.com/ranking/article/117/0003982466
Finished processing https://m.entertain.naver.com/ranking/article/312/0000725795
Finished processing https://m.entertain.naver.com/ranking/article/477/0000566878
Finished processing https://m.entertain.naver.com/ranking/article/415/0000040866
Finished processing https://m.entertain.naver.com/ranking/article/052/0002241108
Finished processing https://